<a href="https://colab.research.google.com/github/sekshem/-Prompt-Engineering-with-Groq-API-LLMs/blob/main/prompt_engineering_groq_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Engineering with Groq API LLMs — Hands-on Colab Exercise

This notebook covers:
- Zero-shot, one-shot, few-shot
- Role/persona prompting
- Instruction prompting
- Context-grounded prompting
- JSON / structured output prompting
- Classification, extraction, summarization, rewriting
- Reasoning-style prompting, step-back prompting
- Self-critique, prompt chaining, multi-turn prompting
- Rubric-based evaluation and prompt templates

In [ ]:
print("Prompt Engineering with Groq API LLMs - Hands-on Colab Exercise")
print("We will test multiple prompt engineering techniques cell by cell.")

In [ ]:
!pip install -q groq pandas tabulate

In [ ]:
import os
import json
import textwrap
import pandas as pd
from groq import Groq
from tabulate import tabulate

In [ ]:
import getpass
GROQ_API_KEY = getpass.getpass("Enter your Groq API Key: ")
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
MODEL_NAME = "llama-3.3-70b-versatile"
print("Using model:", MODEL_NAME)

In [ ]:
def ask_groq(prompt, system_prompt="You are a helpful AI assistant.", model=MODEL_NAME, temperature=0.3, max_tokens=1024):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [ ]:
def show_result(title, prompt, response, width=110):
    print("="*120)
    print(f"TECHNIQUE: {title}")
    print("-"*120)
    print("PROMPT:")
    print(textwrap.fill(prompt, width=width))
    print("-"*120)
    print("MODEL RESPONSE:")
    print(response)
    print("="*120)

In [ ]:
customer_feedback = """
I liked the app interface and the speed of checkout. However, the payment page failed twice before succeeding.
Customer support replied after 2 days, which is too slow. Product quality was good overall, but delivery packaging was damaged.
I may still recommend it because the discounts were attractive.
"""
print(customer_feedback)

In [ ]:
data = [
    {"feedback_id": 101, "feedback": "Checkout was fast and the UI is clean, but payment failed once and support was slow."},
    {"feedback_id": 102, "feedback": "Good product quality, nice discount, but damaged packaging and late delivery."},
    {"feedback_id": 103, "feedback": "The app is intuitive and easy to use. Customer support resolved my issue quickly."}
]
df = pd.DataFrame(data)
df

In [ ]:
prompt_zero_shot = f"""
Read the customer feedback below and identify:
1. Positive points
2. Negative points
3. Final customer sentiment as Positive / Neutral / Negative

Feedback:
{customer_feedback}
"""
response_zero_shot = ask_groq(prompt_zero_shot)
show_result("Zero-shot Prompting", prompt_zero_shot, response_zero_shot)

In [ ]:
prompt_one_shot = f"""
Example:
Feedback: "The app design is nice but delivery was late."
Output:
Positive points: app design
Negative points: late delivery
Sentiment: Neutral

Now do the same for the feedback below.

Feedback:
{customer_feedback}
"""
response_one_shot = ask_groq(prompt_one_shot)
show_result("One-shot Prompting", prompt_one_shot, response_one_shot)

In [ ]:
prompt_few_shot = f"""
You are analyzing e-commerce customer feedback.

Example 1:
Feedback: "The product quality is excellent but shipping was delayed."
Output:
Positive points: product quality
Negative points: delayed shipping
Sentiment: Neutral

Example 2:
Feedback: "Support was quick, refund was smooth, and the app was easy to use."
Output:
Positive points: quick support, smooth refund, easy app usage
Negative points: none
Sentiment: Positive

Example 3:
Feedback: "Payment failed twice and the package arrived broken."
Output:
Positive points: none
Negative points: payment failure, damaged package
Sentiment: Negative

Now analyze this feedback:

Feedback:
{customer_feedback}
"""
response_few_shot = ask_groq(prompt_few_shot)
show_result("Few-shot Prompting", prompt_few_shot, response_few_shot)

In [ ]:
prompt_role = f"""
You are a senior Customer Experience Analyst in an e-commerce company.
Analyze the feedback and provide:
1. Customer pain points
2. Business risks
3. Improvement recommendations

Feedback:
{customer_feedback}
"""
response_role = ask_groq(prompt_role)
show_result("Role / Persona Prompting", prompt_role, response_role)

In [ ]:
prompt_instruction = f"""
Task: Analyze the feedback.

Rules:
- Use bullet points
- Keep each bullet under 15 words
- Separate positives, negatives, sentiment, and action items
- Do not add information not present in the feedback

Feedback:
{customer_feedback}
"""
response_instruction = ask_groq(prompt_instruction)
show_result("Instruction-based Prompting", prompt_instruction, response_instruction)

In [ ]:
prompt_context = f"""
Context:
You are working for an e-commerce platform whose priorities are:
1. Reduce payment failures
2. Improve customer support turnaround time
3. Reduce packaging damage complaints

Task:
Analyze the feedback in the context of these business priorities and rank the top 3 issues.

Feedback:
{customer_feedback}
"""
response_context = ask_groq(prompt_context)
show_result("Context-grounded Prompting", prompt_context, response_context)

In [ ]:
prompt_json = f"""
Analyze the customer feedback and return output in valid JSON only.

Required schema:
{
  "positive_points": [],
  "negative_points": [],
  "sentiment": "",
  "recommendation_priority": []
}

Feedback:
{customer_feedback}
"""
response_json = ask_groq(prompt_json)
show_result("Structured / JSON Output Prompting", prompt_json, response_json)

In [ ]:
try:
    parsed = json.loads(response_json)
    print(json.dumps(parsed, indent=2))
except Exception as e:
    print("JSON parsing failed:", e)
    print("Raw output:")
    print(response_json)

In [ ]:
prompt_classification = f"""
Classify the feedback into one of these categories:
- Product Issue
- Delivery Issue
- Support Issue
- Payment Issue
- UX/UI Praise
- Mixed Experience

Return:
1. Primary category
2. Secondary category if any
3. Reason in 2 lines

Feedback:
{customer_feedback}
"""
response_classification = ask_groq(prompt_classification)
show_result("Classification Prompting", prompt_classification, response_classification)

In [ ]:
prompt_extraction = f"""
Extract the following from the feedback:
- product_quality_issue: yes/no
- delivery_issue: yes/no
- support_issue: yes/no
- payment_issue: yes/no
- discount_mentioned: yes/no
- recommendation_intent: yes/no

Feedback:
{customer_feedback}

Return only JSON.
"""
response_extraction = ask_groq(prompt_extraction)
show_result("Information Extraction Prompting", prompt_extraction, response_extraction)

In [ ]:
prompt_summary = f"""
Summarize the feedback in:
1. 1 sentence
2. 3 bullet points
3. an executive summary under 50 words

Feedback:
{customer_feedback}
"""
response_summary = ask_groq(prompt_summary)
show_result("Summarization Prompting", prompt_summary, response_summary)

In [ ]:
prompt_rewrite = f"""
Rewrite the following feedback into:
1. A professional escalation email to the operations team
2. A concise CRM note
3. A tweet-sized summary

Feedback:
{customer_feedback}
"""
response_rewrite = ask_groq(prompt_rewrite)
show_result("Transformation / Rewriting Prompting", prompt_rewrite, response_rewrite)

In [ ]:
prompt_reasoning = f"""
Analyze the customer feedback step by step.

Return:
1. Key observations
2. How you inferred sentiment
3. Final conclusion

Keep reasoning concise and task-focused.

Feedback:
{customer_feedback}
"""
response_reasoning = ask_groq(prompt_reasoning)
show_result("Reasoning-style Prompting", prompt_reasoning, response_reasoning)

In [ ]:
prompt_step_back = f"""
Before analyzing the feedback, first answer:
"What are the common dimensions used to evaluate customer feedback in e-commerce?"

Then use those dimensions to analyze the following feedback:
{customer_feedback}

Return:
1. General evaluation dimensions
2. Feedback analysis using those dimensions
"""
response_step_back = ask_groq(prompt_step_back)
show_result("Step-back Prompting", prompt_step_back, response_step_back)

In [ ]:
initial_prompt = f"""
Analyze the feedback and provide positives, negatives, sentiment, and action items.

Feedback:
{customer_feedback}
"""
initial_response = ask_groq(initial_prompt)

critique_prompt = f"""
You previously generated the following analysis:

{initial_response}

Now critique it for:
1. Missing points
2. Overstatements
3. Weak recommendations
4. Better final version

Original feedback:
{customer_feedback}
"""
critique_response = ask_groq(critique_prompt)
show_result("Initial Analysis", initial_prompt, initial_response)
show_result("Self-reflection / Critique Prompting", critique_prompt, critique_response)

In [ ]:
prompt_chain_1 = f"""
Extract all issues mentioned in the feedback as a bullet list.

Feedback:
{customer_feedback}
"""
issues_output = ask_groq(prompt_chain_1)
show_result("Prompt Chaining - Stage 1", prompt_chain_1, issues_output)

In [ ]:
prompt_chain_2 = f"""
Given these extracted issues:
{issues_output}

Rank them by business severity from highest to lowest.
Explain each ranking in one line.
"""
priority_output = ask_groq(prompt_chain_2)
show_result("Prompt Chaining - Stage 2", prompt_chain_2, priority_output)

In [ ]:
prompt_chain_3 = f"""
Using this prioritized issue list:
{priority_output}

Create a 30-day action plan for the operations team.
"""
action_plan_output = ask_groq(prompt_chain_3)
show_result("Prompt Chaining - Stage 3", prompt_chain_3, action_plan_output)

In [ ]:
def chat_groq(messages, model=MODEL_NAME, temperature=0.3, max_tokens=1024):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=messages
    )
    return response.choices[0].message.content

In [ ]:
messages = [
    {"role": "system", "content": "You are a customer feedback analyst."},
    {"role": "user", "content": f"Analyze this feedback:\n{customer_feedback}"}
]

resp1 = chat_groq(messages)
print("Assistant:", resp1)

messages.append({"role": "assistant", "content": resp1})
messages.append({"role": "user", "content": "Now convert that into a priority matrix with High / Medium / Low severity."})

resp2 = chat_groq(messages)
print("\nAssistant Follow-up:\n", resp2)

In [ ]:
candidate_answer = """
Positive points: app UI, checkout speed, discounts
Negative points: payment page failed, support was slow, packaging damaged
Sentiment: Neutral
"""

prompt_eval = f"""
Evaluate the following answer on a scale of 1 to 5 for each criterion:
- Completeness
- Accuracy
- Relevance
- Clarity

Also provide:
- total score /20
- 3 improvement suggestions

Original feedback:
{customer_feedback}

Candidate answer:
{candidate_answer}
"""
response_eval = ask_groq(prompt_eval)
show_result("Rubric-based Evaluation Prompting", prompt_eval, response_eval)

In [ ]:
def build_feedback_prompt(feedback, task_type="summary"):
    templates = {
        "summary": f"Summarize the following feedback in 3 bullet points:\n{feedback}",
        "sentiment": f"Identify sentiment and explain why:\n{feedback}",
        "action_items": f"Read the feedback and propose action items:\n{feedback}",
        "json_extract": f"""
Extract issues from this feedback and return JSON with:
payment_issue, support_issue, delivery_issue, packaging_issue, sentiment

Feedback:
{feedback}
"""
    }
    return templates.get(task_type, templates["summary"])

In [ ]:
for task in ["summary", "sentiment", "action_items", "json_extract"]:
    p = build_feedback_prompt(customer_feedback, task)
    r = ask_groq(p)
    show_result(f"Prompt Template: {task}", p, r)

In [ ]:
comparison_prompts = {
    "zero_shot": f"Analyze the sentiment of this feedback:\n{customer_feedback}",
    "role_based": f"You are a CX manager. Analyze sentiment and business risks:\n{customer_feedback}",
    "json_output": f"""
Return JSON with sentiment, positives, negatives for:
{customer_feedback}
""",
    "few_shot": f"""
Example:
Feedback: Delivery was late but the product was good.
Sentiment: Neutral

Now analyze:
{customer_feedback}
"""
}

results = []
for name, pr in comparison_prompts.items():
    out = ask_groq(pr)
    results.append({"prompt_type": name, "response_preview": out[:300]})

comparison_df = pd.DataFrame(results)
comparison_df

In [ ]:
capstone_text = """
A retail company has collected 500 customer reviews from app, email, and support tickets.
Management wants:
1. top complaint categories,
2. overall sentiment,
3. urgent operational issues,
4. a concise weekly CX report.
"""
print(capstone_text)

In [ ]:
prompt_capstone = f"""
You are a Lead Customer Experience Analyst.

Problem:
{capstone_text}

Using the sample feedback below as if it were part of the weekly dataset:
{customer_feedback}

Prepare a weekly CX insight report with:
1. Top issues
2. Positive themes
3. Risk areas
4. Immediate actions
5. Suggested dashboard KPIs
"""
response_capstone = ask_groq(prompt_capstone)
show_result("Capstone Prompt Engineering Exercise", prompt_capstone, response_capstone)

In [ ]:
def analyze_feedback_row(feedback):
    prompt = f"""
Analyze this customer feedback and return JSON with:
- sentiment
- positives
- negatives
- main_issue_category

Feedback:
{feedback}
"""
    return ask_groq(prompt)

df["llm_analysis"] = df["feedback"].apply(analyze_feedback_row)
df

In [ ]:
reflection_questions = [
    "Which prompting technique gave the most structured output?",
    "Which prompt reduced hallucination risk the most?",
    "Which technique is best for analytics pipelines?",
    "Which technique is best for chatbots?",
    "How did output quality change from zero-shot to few-shot?",
    "When should prompt chaining be preferred over one big prompt?"
]

for i, q in enumerate(reflection_questions, 1):
    print(f"{i}. {q}")